# Kuramoto--Sivashinsky with Neumann boundary conditions

This notebook is the fixed-boundary, computer-assisted-proof companion to `ks_periodic_forecasting.ipynb`. The original notebook solves the velocity KS equation on a periodic domain and searches for relative periodic orbits. Here the potential satisfies

\[h_t=-\tfrac12h_x^2-h_{xx}-h_{xxxx}+\langle\tfrac12h_x^2\rangle_x,\qquad h_x=h_{xxx}=0\quad(x=0,\ell),\]

with zero spatial mean. Thus `u=h_x` satisfies the standard velocity KS equation with `u=u_xx=0` at the endpoints. Cosines represent `h`, sines represent `u`, and the odd velocity extension has period `2 ell`. Fixed endpoints remove generic continuous spatial translations (only a discrete reflection/sign symmetry remains), so this notebook uses an ordinary periodic orbit, no drift variable, and one temporal phase condition.


In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = next(path for path in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (path / 'CAP').exists())
CAP_DIR = REPO_ROOT / 'CAP/Kuramoto_PO'
OUTPUT_DIR = CAP_DIR / 'outputs_surrogate'
sys.path.insert(0, str(CAP_DIR))

from ks_neumann import (
    NeumannKSConfig, boundary_defects, independent_flow_closure,
    load_candidate, oversampled_residual, reconstruct_potential,
    reconstruct_velocity, spatial_wavenumbers,
)

CONFIG = NeumannKSConfig(interval_length=22.0, spatial_modes=60, temporal_modes=160, full_grid_nodes=240)
RUN_CANDIDATE_SEARCH = False
RUN_INDEPENDENT_FLOW_CHECK = False
RUN_FULL_PDE_PROOF = False


## Surrogate-only candidate construction

Independent random spatial snapshots and exact spectral derivative labels fit complete quadratic surrogate ODEs. Reflection equivariance is enforced; sine coordinates are invertibly related to interior nodal values. A 16-mode **surrogate trajectory** supplies the recurrence. All subsequent orbit solves use fitted surrogates at 16–60 spatial modes, up to 160 temporal harmonics. No true-system periodic orbit is loaded or refined. Exact reality and half-period symmetry are enforced during export. The true-system optimizer count is zero over the entire workflow.

Training uses exact derivatives, not noisy scalar measurements. Enable regeneration below only when needed; artifacts are stored separately from the legacy true-Galerkin experiment.


In [ ]:
if RUN_CANDIDATE_SEARCH:
    subprocess.run(
        [sys.executable, str(CAP_DIR / 'generate_surrogate_candidate.py'), '--sample-time', '800'],
        cwd=CAP_DIR, check=True,
    )
else:
    print('Using the existing frozen candidate. Set RUN_CANDIDATE_SEARCH=True to regenerate it.')


In [ ]:
display(pd.read_csv(OUTPUT_DIR / 'surrogate_fit_metrics.csv'))
display(pd.read_csv(OUTPUT_DIR / 'surrogate_continuation_stages.csv'))


## Resolution, boundary, and ordinary-periodicity diagnostics

The oversampled residual is evaluated away from the collocation nodes so unresolved quadratic temporal modes cannot alias into an apparently exact residual. The independent ETDRK4 closure is a nonrigorous diagnostic and can be large for an unstable orbit even when the local invariance residual is small.


In [ ]:
candidate_path = OUTPUT_DIR / 'ks_neumann_cap_candidate.npz'
candidate = load_candidate(candidate_path)
time_values = candidate['velocity_time_values']
frequency = float(candidate['frequency'])
period = float(candidate['period'])
_, residual = oversampled_residual(time_values, frequency, CONFIG)
print({'period': period,
       'cap_transfer_newton_corrections': int(candidate['newton_corrections']),
       'oversampled_residual_rms': float(np.sqrt(np.mean(residual**2))),
       'oversampled_residual_max': float(np.max(np.abs(residual))),
       **boundary_defects(time_values, CONFIG)})
if RUN_INDEPENDENT_FLOW_CHECK:
    print(independent_flow_closure(time_values, period, CONFIG))


In [ ]:
phase_indices = [0, len(time_values)//4, len(time_values)//2, 3*len(time_values)//4]
x = np.linspace(0.0, CONFIG.interval_length, 500)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for index in phase_indices:
    axes[0].plot(x, reconstruct_velocity(time_values[index], x, CONFIG), label=f'phase {index/len(time_values):.2f}')
    axes[1].plot(x, reconstruct_potential(time_values[index], x, CONFIG))
axes[0].set(title='Velocity u=h_x', xlabel='x', ylabel='u')
axes[1].set(title='Mean-zero potential h', xlabel='x', ylabel='h')
axes[0].legend()
fig.tight_layout()
plt.show()


## Infinite-dimensional INTLAB radii-polynomial proof

The proof uses a 6421-dimensional, symmetry-reduced space--time Fourier inverse and a diagonal raw-linear inverse on the infinite tail. The preprocessing stage exhausts every finite/tail coupling column through the complete convolution halo and supplies analytic bounds outside it. MATLAB/INTLAB then recomputes the de-aliased residual with outward rounding and checks the final radii polynomial. The frozen candidate receives zero Newton corrections.


In [ ]:
if RUN_FULL_PDE_PROOF:
    completed = subprocess.run(
        ['bash', str(CAP_DIR / 'run_surrogate_pde_proof.sh')],
        cwd=REPO_ROOT, text=True, capture_output=True, check=True,
    )
    print(completed.stdout[-5000:])
summary_path = OUTPUT_DIR / 'ks_neumann_cap_summary.csv'
if summary_path.exists():
    display(pd.read_csv(summary_path))
else:
    print(f'CAP summary not present yet: {summary_path}')
